In [ ]:
#import cleaned data
import pandas as pd

df = pd.read_csv("../data/processed/clean_trials.csv")

print(df.shape)
df.head()

In [ ]:
print(df['phase'].head(10))
print(df['phase'].unique()[:10])

In [ ]:
# --- Percentile-based RBM Risk Flags ---

# High Enrollment Risk (top 25%)
enrollment_threshold = df['enrollment'].quantile(0.75)
df['high_enrollment_flag'] = (df['enrollment'] > enrollment_threshold).astype(int)

# Multi-site Risk (top 25%)
location_threshold = df['locations'].quantile(0.75)
df['multi_site_flag'] = (df['locations'] > location_threshold).astype(int)

# Long Duration Risk (top 25%)
duration_threshold = df['study_duration_days'].quantile(0.75)
df['long_duration_flag'] = (df['study_duration_days'] > duration_threshold).astype(int)

In [ ]:
# --- Clean Phase Column ---
import ast

def clean_phase(x):
    # Case 1: already clean string like "PHASE3"
    if isinstance(x, str) and not x.startswith("["):
        return x.strip()

    # Case 2: stringified list like "['PHASE3']"
    if isinstance(x, str) and x.startswith("["):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list) and len(parsed) > 0:
                return parsed[0]
        except:
            return "NA"

    # Case 3: actual list
    if isinstance(x, list) and len(x) > 0:
        return x[0]

    return "NA"


df['phase'] = df['phase'].apply(clean_phase)

print(df['phase'].value_counts())

In [ ]:
# Create RBM Phase risk map
phase_risk_map = {
    "EARLY_PHASE1": 4,
    "PHASE1": 4,
    "PHASE2": 3,
    "PHASE3": 3,
    "PHASE4": 1,
    "NA": 2
}

df['phase_risk'] = df['phase'].map(phase_risk_map).fillna(2)

In [ ]:
# Create RBM feature - Invervention risk
intervention_risk_map = {
    "DRUG": 3,
    "BIOLOGICAL": 3,
    "DEVICE": 2,
    "PROCEDURE": 2,
    "BEHAVIORAL": 1,
    "DIAGNOSTIC_TEST": 1,
    "OTHER": 1
}

df['intervention_risk'] = df['intervention_type'].map(intervention_risk_map).fillna(1)

In [ ]:
# Location data availability (data quality signal)
df['has_location_data'] = (df['locations'] > 0).astype(int)

# Check distribution
print(df['has_location_data'].value_counts(normalize=True))

In [ ]:
# Total Risk Score
df['total_risk_score'] = (
    df['high_enrollment_flag'] +
    df['multi_site_flag'] +
    df['long_duration_flag'] +
    df['phase_risk'] +
    df['intervention_risk'] +
    (1 - df['has_location_data'])
)

In [ ]:
# --- RBM FEATURE SUMMARY ---

print("=== RBM FLAG DISTRIBUTIONS ===\n")

print("High Enrollment Flag:")
print(df['high_enrollment_flag'].value_counts(normalize=True), "\n")

print("Multi-site Flag:")
print(df['multi_site_flag'].value_counts(normalize=True), "\n")

print("Long Duration Flag:")
print(df['long_duration_flag'].value_counts(normalize=True), "\n")

print("Has Location Data:")
print(df['has_location_data'].value_counts(normalize=True), "\n")


print("=== RISK SCORE COMPONENTS ===\n")

print("Phase Risk Distribution:")
print(df['phase_risk'].value_counts(normalize=True), "\n")

print("Intervention Risk Distribution:")
print(df['intervention_risk'].value_counts(normalize=True), "\n")


print("=== TOTAL RISK SCORE SUMMARY ===\n")

print(df['total_risk_score'].describe(), "\n")

print("Top 10 Highest Risk Trials:")
print(
    df.sort_values('total_risk_score', ascending=False)[
        ['nct_id', 'phase', 'intervention_type', 'total_risk_score']
    ].head(10)
)

print("\nAverage Risk Score by Intervention Type:")
print(df.groupby('intervention_type')['total_risk_score'].mean().sort_values(ascending=False))

In [ ]:
# Save the final dataset with engineered features
import os

output_path = "/Users/suhailbarakzai/Documents/Clinical Trial Data Analysis/projects/02_clinical_trial_rbm_analysis/data/processed/rbm_feature_engineered_trials.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False)

print("Saved final RBM dataset to:", output_path)